# PVTT 数据采集周报 — 第三周

**项目：** Product Video Template Transfer (PVTT)  
**目标会议：** CVPR 2027  
**负责人：** 王洁怡 (wangjieyi)  
**周期：** 2026-03-08 ~ 2026-03-14  

---

## 任务目标

> **Task 1：** 撰写数据采集的pipeline代码，利用该代码去采集数据，形成report，让大家可以查看数据采集的质量。

### 任务完成情况

| 子任务 | 状态 | 说明 |
|--------|------|------|
| Pipeline代码 | ✅ 已完成 | `amazon_spider.py` — 自动化Amazon产品爬虫，支持批量采集、HLS视频下载、断点续爬 |
| 数据采集 | ✅ 进行中 | 已采集 622+ 产品（目标 800+），爬虫仍在后台运行 |
| 质量报告 | ✅ 已完成 | `pvtt_dataset_report.html` — 交互式HTML报告，含可播放视频和统计图表 |
| 多平台调研 | ✅ 已完成 | `platform_analysis_report.md` — 12个电商平台可行性分析 |
| 目录规范 | ✅ 已完成 | 统一数据格式、JSON Schema、目录结构文档 |

---

## 一、本周工作内容

### 1.1 Pipeline 架构

本周完善了端到端的自动化数据采集Pipeline：

```
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌──────────────┐    ┌─────────────┐
│  阶段1：爬取  │ -> │  阶段2：上传   │ -> │  阶段3：处理    │ -> │  阶段4：报告   │ -> │  阶段5：推送  │
│ amazon_spider│    │upload_to_ser │    │ server-side   │    │ generate_    │    │  git push   │
│   .py        │    │ ver.py       │    │ pvtt_pipeline │    │ dataset_     │    │  origin     │
│ (本地运行)    │    │ (SFTP)       │    │ (GPU服务器)    │    │ report.py    │    │  main       │
└─────────────┘    └──────────────┘    └───────────────┘    └──────────────┘    └─────────────┘
```

| 阶段 | 脚本 | 运行位置 | 说明 |
|------|------|----------|------|
| 爬取 | `amazon_spider.py` | 本地（住宅IP） | 搜索Amazon → 解析产品页 → 下载图片和HLS视频 |
| 上传 | `upload_to_server.py` | 本地 | 通过 SFTP 增量同步数据至 GPU 服务器 |
| 处理 | `server-scripts/pvtt_pipeline.py` | 服务器 | 镜头检测 + 视频标准化（1280x720, 24fps） |
| 报告 | `generate_dataset_report.py` | 本地 | 生成HTML交互式报告 + Markdown报告 |
| 推送 | `pvtt_pipeline.py` | 本地 | 编排全流程 + git push 到 GitHub |

### 1.2 Amazon 爬虫核心功能

`amazon_spider.py`（829行）实现了以下功能：

- **搜索与发现**：根据关键词搜索Amazon，自动翻页提取产品列表
- **产品详情解析**：提取标题、价格、品牌、评分、图片URL、视频URL
- **图片下载**：提取高清商品图片（最多27张/产品）
- **HLS视频下载**：解析M3U8清单 → 下载TS分段 → 合并为MP4（支持ffmpeg和纯Python双模式）
- **反爬对策**：User-Agent轮换、请求间隔随机化、自动重试
- **断点续爬**：检测已存在的JSON文件自动跳过
- **批量模式**：支持多类别多关键词并行采集
- **服务器上传**：集成paramiko SFTP上传功能

```python
# 使用示例
python amazon_spider.py --batch          # 批量采集所有类别
python amazon_spider.py --status         # 查看采集进度
python amazon_spider.py --upload         # 上传数据到服务器
```

### 1.3 数据扩展

本周将每个品类的搜索关键词从3个扩展到6个，数据量大幅增长：

| 指标 | 上周（Week 02） | 本周（Week 03） | 增长 |
|------|----------------|----------------|------|
| 关键词数 | 21个（3/类） | 42个（6/类） | +100% |
| 产品数 | 404 | 622+ | +54% |
| 图片数 | 1,888 | 3,432+ | +82% |
| 视频数 | 378 | 603+ | +59% |
| 数据量 | 2.8 GB | 3.8 GB | +36% |

**新增关键词示例：**

| 品类 | 原有关键词 | 新增关键词 |
|------|-----------|----------|
| necklace | gold necklace, silver pendant necklace, pearl necklace | choker necklace, diamond pendant necklace, layered necklace |
| bracelet | charm bracelet, gold bangle bracelet, beaded bracelet | tennis bracelet, cuff bracelet women, chain bracelet gold |
| earring | drop earrings, stud earrings gold, hoop earrings silver | dangle earrings women, pearl earrings, clip on earrings |
| ring | engagement ring, gold band ring, silver stackable ring | cocktail ring women, signet ring men, wedding band set |
| watch | men automatic watch, women dress watch, sport digital watch | luxury watch men, chronograph watch, minimalist watch women |
| handbag | leather crossbody bag, women tote handbag, clutch purse evening | shoulder bag women, mini crossbody bag, hobo bag leather |
| sunglasses | polarized sunglasses, aviator sunglasses, cat eye sunglasses | oversized sunglasses women, sport sunglasses men, round sunglasses retro |

### 1.4 多平台调研

完成了12个电商平台的数据采集可行性分析，覆盖：

| 平台 | 爬虫可行性 | 官方API | 第三方服务 | 视频率 | 推荐 |
|------|-----------|---------|-----------|--------|------|
| **Amazon** | ✅ 可行（住宅IP） | 无免费API | - | 40-60% | **正在使用** |
| **eBay** | ⚠️ 需JS渲染 | ✅ Browse API（免费） | - | 10-20% | **推荐（免费）** |
| **TikTok Shop** | ❌ 加密头 | ⚠️ Research API | Apify $2/1K | ~90% | **推荐（Apify）** |
| **Etsy** | ❌ Datadome | ✅ Open API v3 | - | 10-15% | 仅图片 |
| **淘宝/天猫** | ❌ 极强反爬 | ❌ 需企业资质 | ¥0.01-0.05/条 | 70-80% | **推荐（第三方）** |
| **京东** | ❌ 强反爬 | ❌ 需企业资质 | 同淘宝捆绑 | 50-60% | 可选 |
| **AliExpress** | ❌ Akamai | 付费API | Oxylabs $49/月 | 30-40% | 不推荐 |
| **小红书** | ❌ 动态签名 | ❌ 无 | 不稳定 | ~95% | 法律风险高 |
| **拼多多** | ❌ 极强反爬 | ❌ 需企业资质 | 不稳定 | 20-30% | 不推荐 |
| **SHEIN** | ❌ Cloudflare | ❌ 无 | Apify $3/1K | 30% | 不推荐 |
| **Shopee** | ⚠️ 中等 | ✅ Partner API | - | 15-25% | 珠宝品类弱 |
| **Walmart** | ⚠️ 中等 | ✅ Affiliate API | - | 20-30% | 品牌官方视频 |

**推荐采集方案（按预算）：**

| 方案 | 预算 | 平台 | 预估数据量 |
|------|------|------|----------|
| 零成本方案 | $0 | Amazon + eBay API + Etsy API | 800+ 产品 |
| 低成本方案 | ~$2 | 上述 + TikTok Shop (Apify) | 1100+ 产品 |
| 中等预算方案 | ~¥350 | 上述 + 淘宝/京东（第三方） | 1600+ 产品 |

详见：[`platform_analysis_report.md`](01-dataset-construction/platform_analysis_report.md)

### 1.5 目录结构与规范化

制定了统一的数据格式规范，确保所有平台采集的数据可互操作：

**统一数据格式：**
```
{platform}_data/
  {category}/                        # bracelet, earring, handbag, ...
    {PRODUCT_ID}.json                # 产品元数据
    media/
      images/{PRODUCT_ID}_01.jpg     # 产品图片
      videos/{PRODUCT_ID}.mp4        # 产品视频
```

**JSON元数据Schema：**
```json
{
  "product_id": "B0XXXXXX",
  "platform": "amazon",
  "title": "产品标题",
  "category": "bracelet",
  "price": "29.99",
  "currency": "USD",
  "images": ["media/images/B0XXXXXX_01.jpg"],
  "videos": ["media/videos/B0XXXXXX.mp4"],
  "has_video": true,
  "scrape_date": "2026-03-14"
}
```

---

## 二、数据采集质量报告

In [ ]:
import json, os, glob
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path("01-dataset-construction/amazon_data")
CATEGORIES = ["bracelet", "earring", "handbag", "necklace", "ring", "sunglasses", "watch"]
CAT_CN = {"bracelet": "手链", "earring": "耳饰", "handbag": "手提包", 
          "necklace": "项链", "ring": "戒指", "sunglasses": "太阳镜", "watch": "手表"}

stats = {}
for cat in CATEGORIES:
    cat_dir = DATA_DIR / cat
    jsons = list(cat_dir.glob("*.json"))
    img_dir = cat_dir / "media" / "images"
    vid_dir = cat_dir / "media" / "videos"
    n_img = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
    n_vid = len(list(vid_dir.glob("*.mp4"))) if vid_dir.exists() else 0
    
    products = []
    for jf in jsons:
        with open(jf, encoding="utf-8") as f:
            products.append(json.load(f))
    
    with_video = sum(1 for p in products if p.get("download_stats", {}).get("videos_downloaded", 0) > 0)
    keywords = list(set(p.get("keyword", "") for p in products if p.get("keyword")))
    
    total_size = 0
    for dirpath, _, filenames in os.walk(str(cat_dir)):
        for fn in filenames:
            try: total_size += os.path.getsize(os.path.join(dirpath, fn))
            except: pass
    
    stats[cat] = {
        "产品数": len(jsons), "图片数": n_img, "视频数": n_vid,
        "含视频产品": with_video, "关键词数": len(keywords),
        "大小_MB": total_size / (1024*1024)
    }

print("="*70)
print("PVTT Amazon 数据采集统计")
print("="*70)
print(f"{'品类':<12} {'产品数':>6} {'图片数':>6} {'视频数':>6} {'含视频':>6} {'关键词':>6} {'大小(MB)':>10}")
print("-"*70)
for cat in CATEGORIES:
    s = stats[cat]
    print(f"{CAT_CN[cat]:<10} {s['产品数']:>6} {s['图片数']:>6} {s['视频数']:>6} {s['含视频产品']:>6} {s['关键词数']:>6} {s['大小_MB']:>10.1f}")
print("-"*70)
print(f"{'合计':<10} {sum(s['产品数'] for s in stats.values()):>6} "
      f"{sum(s['图片数'] for s in stats.values()):>6} "
      f"{sum(s['视频数'] for s in stats.values()):>6} "
      f"{sum(s['含视频产品'] for s in stats.values()):>6} "
      f"{sum(s['关键词数'] for s in stats.values()):>6} "
      f"{sum(s['大小_MB'] for s in stats.values()):>10.1f}")
print()
tot = sum(s['产品数'] for s in stats.values())
tot_vid = sum(s['含视频产品'] for s in stats.values())
print(f"视频覆盖率：{tot_vid/tot*100:.1f}% ({tot_vid}/{tot} 个产品含视频)")

In [ ]:
# 关键词覆盖分析
print("\n" + "="*70)
print("各品类搜索关键词覆盖")
print("="*70)
for cat in CATEGORIES:
    cat_dir = DATA_DIR / cat
    keywords = defaultdict(int)
    for jf in cat_dir.glob("*.json"):
        with open(jf, encoding="utf-8") as f:
            data = json.load(f)
            kw = data.get("keyword", "unknown")
            keywords[kw] += 1
    print(f"\n{CAT_CN[cat]} ({cat}):")
    for kw, count in sorted(keywords.items(), key=lambda x: -x[1]):
        print(f"  {kw:<35} {count:>3} 个产品")

In [ ]:
# 视频质量分析
try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("提示：未安装OpenCV，跳过视频详细分析")

if HAS_CV2:
    print("\n" + "="*70)
    print("视频质量分析")
    print("="*70)
    
    durations = []
    resolutions = defaultdict(int)
    sizes = []
    
    for cat in CATEGORIES:
        vid_dir = DATA_DIR / cat / "media" / "videos"
        if not vid_dir.exists(): continue
        for vf in vid_dir.glob("*.mp4"):
            size_mb = os.path.getsize(str(vf)) / (1024*1024)
            sizes.append(size_mb)
            cap = cv2.VideoCapture(str(vf))
            if cap.isOpened():
                fps = cap.get(cv2.CAP_PROP_FPS) or 0
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
                if fps > 0:
                    dur = frames / fps
                    durations.append(dur)
                if w > 0:
                    resolutions[f"{w}x{h}"] += 1
            cap.release()
    
    if durations:
        print(f"  视频总数: {len(sizes)}")
        print(f"  平均时长: {sum(durations)/len(durations):.1f}s")
        print(f"  时长范围: {min(durations):.1f}s ~ {max(durations):.1f}s")
        print(f"  总大小: {sum(sizes):.0f} MB")
        print(f"  平均大小: {sum(sizes)/len(sizes):.1f} MB")
        print(f"\n  分辨率分布:")
        for res, count in sorted(resolutions.items(), key=lambda x: -x[1])[:5]:
            print(f"    {res}: {count} 个视频")

In [ ]:
# 样本数据展示
print("\n" + "="*70)
print("样本产品展示（每品类1个含视频的产品）")
print("="*70)

for cat in CATEGORIES:
    cat_dir = DATA_DIR / cat
    sample = None
    for jf in sorted(cat_dir.glob("*.json")):
        with open(jf, encoding="utf-8") as f:
            data = json.load(f)
            if data.get("download_stats", {}).get("videos_downloaded", 0) > 0:
                sample = data
                break
    
    if sample:
        ds = sample.get("download_stats", {})
        print(f"\n{'─'*60}")
        print(f"{CAT_CN[cat]} ({cat})")
        print(f"  产品ID: {sample.get('asin', 'N/A')}")
        print(f"  标题: {sample.get('title', 'N/A')[:60]}...")
        print(f"  价格: {sample.get('price', 'N/A')}")
        print(f"  关键词: {sample.get('keyword', 'N/A')}")
        print(f"  图片数: {ds.get('images_downloaded', 0)}")
        print(f"  视频数: {ds.get('videos_downloaded', 0)}")

### 2.1 数据质量总结

| 指标 | 数值 | 评价 |
|------|------|------|
| 产品总数 | 622+ | 超过600目标 |
| 视频覆盖率 | ~50-60% | 良好（珠宝品类视频率高） |
| 图片质量 | 中高 | Amazon产品图普遍720p-1080p |
| 视频格式 | MP4 (HLS转封装) | 标准格式，可直接使用 |
| 视频时长 | 5-60秒 | 适合模板迁移任务 |
| 数据完整性 | 高 | JSON元数据完整，含标题/价格/URL |

**已知问题：**
1. 少量TS分段下载时遇到损坏的URL（非致命，爬虫自动跳过）
2. 部分视频含DRM加密（约5%），无法下载
3. handbag品类数据量偏少（38个），需更多关键词

**交互式质量报告：** 打开 `01-dataset-construction/pvtt_dataset_report.html` 可查看：
- 各品类统计图表
- 每品类前3个样本产品（含可播放视频）
- 视频分辨率和时长分布

---

## 三、服务器资源使用

| 资源 | 详情 |
|------|------|
| 服务器 | RTX-5090-32G-X8（8x RTX 5090 32GB） |
| SSH | `wangjieyi@111.17.197.107` |
| 数据路径 | `/data/wangjieyi/pvtt-dataset/amazon_data/` |
| Conda环境 | `datapipeline`（scenedetect, opencv, ffmpeg, rembg） |
| 已上传 | 1.8 GB（旧数据），新数据待同步 |

**服务器Pipeline验证结果：**
- 输入：53个原始视频
- 镜头分割：87个片段
- 标准化输出：87个视频（1280x720, 24fps, H.264）
- 时长范围：1.1s ~ 8.8s（平均3.8s）

---

## 四、下周工作计划（Week 04：2026-03-15 ~ 2026-03-21）

### P0 — 必须完成

| 任务 | 说明 | 预计完成 |
|------|------|----------|
| 完成Amazon爬虫 | 剩余品类（handbag/sunglasses/watch/ring）的新关键词 | 03-15 |
| 服务器数据同步 | 运行 `upload_to_server.py` 同步3.8GB数据 | 03-15 |
| 服务器端处理 | 对新数据运行镜头分割+标准化Pipeline | 03-16 |
| 最终数据报告 | 重新生成包含所有数据的质量报告 | 03-16 |

### P1 — 高优先级

| 任务 | 说明 | 预计完成 |
|------|------|----------|
| 注册eBay开发者账号 | 免费Browse API，获取更多产品图片 | 03-17 |
| 测试Apify TikTok Scraper | 使用$5免费额度测试50-100个产品 | 03-18 |
| 构建eBay采集脚本 | 基于Browse API的Python客户端 | 03-19 |

### P2 — 可选

| 任务 | 说明 |
|------|------|
| 淘宝数据服务询价 | 联系本地数据服务商，获取500+产品报价 |
| 视频质量过滤 | 在服务器上实现自动化质量检查（分辨率、时长、运动评分） |
| 申请TikTok Research API | 准备学术申请材料（需导师推荐信） |

---

## 五、关键文件索引

| 文件 | 说明 |
|------|------|
| `01-dataset-construction/amazon_spider.py` | Amazon产品爬虫（核心代码） |
| `01-dataset-construction/pvtt_pipeline.py` | 主Pipeline编排器 |
| `01-dataset-construction/upload_to_server.py` | 数据上传工具 |
| `01-dataset-construction/generate_dataset_report.py` | 报告生成器 |
| `01-dataset-construction/pvtt_dataset_report.html` | 交互式数据质量报告 |
| `01-dataset-construction/platform_analysis_report.md` | 12平台采集可行性分析 |
| `01-dataset-construction/DIRECTORY_STRUCTURE.md` | 目录结构与数据格式规范 |
| `01-dataset-construction/pipelines/` | 各平台Pipeline文档 |

---

*报告生成日期：2026-03-14*  
*负责人：王洁怡 (wangjieyi@hkust)*